# 🛰️ AstraGuard AI — Spacecraft Telemetry Anomaly Detection
## Machine Learning Pipeline Analysis Notebook

---

**IBM AI Builders Challenge — August 2026**  
**Theme: Advance Space Exploration with Artificial Intelligence**

---

### ⚠️ Prototype Disclaimer

> **All telemetry data used in this notebook is entirely SIMULATED.**  
> It was procedurally generated for demonstration and research purposes only.  
> This is **NOT** real NASA, ESA, ISRO, or any other agency spacecraft telemetry.  
> AstraGuard AI is a prototype decision-support system and must **NOT** be used  
> for actual spacecraft operations, safety-critical decisions, or mission planning.

---

**Author:** AstraGuard AI Team  
**Project Repository:** AstraGuard AI  
**Stack:** Python · scikit-learn · pandas · Streamlit · IBM Watson


## 1. Project Objective

### Why Spacecraft Telemetry Monitoring Matters

Spacecraft continuously generate streams of sensor data — temperature readings from instrument bays, battery voltage levels, power consumption across subsystems, radiation dosimetry, communication signal strength, fuel levels, and solar array output. In real missions, anomalies in these channels can signal imminent hardware failures, environmental hazards, or subsystem faults that, if undetected, may compromise instrument health, mission objectives, or vehicle safety.

Traditional threshold-based monitoring raises alerts only when individual channels cross fixed limits. This misses **multivariate anomalies** — situations where no single channel is out of range, but the *combination* of readings is statistically unusual and operationally significant.

### Why Unsupervised Machine Learning is Appropriate

Labelled anomaly datasets are extremely rare in real-world spacecraft operations — teams cannot pre-label every possible fault mode. **Unsupervised anomaly detection** is therefore the natural choice:

- No labelled training data is required.
- The model learns the statistical structure of normal operations.
- Deviations from learned normality are surfaced as anomalies.
- Novel fault patterns not seen during design can still be detected.

### How AstraGuard AI Addresses This

AstraGuard AI uses an **IsolationForest** anomaly detector trained on seven simulated telemetry channels. The model produces a continuous **anomaly score** and a binary **anomaly flag** for every telemetry reading. These outputs feed a **Mission Risk Engine** that translates ML detections into an explainable **Mission Risk Index (MRI)** on a 0–100 scale, enabling operators to prioritise investigation without requiring deep ML expertise.


## 2. Import Libraries

We import only the libraries that are genuinely needed by this analysis. The project's own source modules in `src/` are added to the Python path so we can reuse the existing implementations directly.

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Numeric / data ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates

matplotlib.rcParams.update({
    "figure.facecolor":  "#0a0e1a",
    "axes.facecolor":    "#111827",
    "axes.edgecolor":    "#1e3a5f",
    "axes.labelcolor":   "#94a3b8",
    "text.color":        "#e2e8f0",
    "xtick.color":       "#64748b",
    "ytick.color":       "#64748b",
    "grid.color":        "#1e3a5f",
    "grid.linestyle":    "--",
    "grid.alpha":        0.5,
    "font.family":       "DejaVu Sans",
    "figure.titlesize":  14,
    "axes.titlesize":    11,
    "axes.labelsize":    9,
})

# ── ML ────────────────────────────────────────────────────────────────────────
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── Project source path ───────────────────────────────────────────────────────
# Resolve repo root whether notebook is run from repo root or notebooks/
REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repo root : {REPO_ROOT}")
print(f"src/       : {SRC_DIR}")
print(f"Python    : {sys.version.split()[0]}")
print(f"pandas    : {pd.__version__}")
print(f"numpy     : {np.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"joblib    : {joblib.__version__}")

## 3. Load the Simulated Telemetry Dataset

The dataset was generated by `generate_telemetry.py` and contains **1,000 five-minute telemetry readings** spanning the year 2026. Approximately 5% of records contain injected anomalies simulating realistic spacecraft faults such as battery undervoltage, thermal spikes, radiation bursts, and signal fades.

We use the project's own `load_telemetry()` function from `src/anomaly_detector.py` to ensure consistency with the production pipeline.

In [ ]:
from anomaly_detector import (
    load_telemetry,
    load_model,
    score_dataframe,
    detection_summary,
    FEATURE_COLS,
    IF_PARAMS,
    DEFAULT_DATA_PATH,
    DEFAULT_MODEL_PATH,
)

DATA_PATH  = REPO_ROOT / "data"   / "telemetry.csv"
MODEL_PATH = REPO_ROOT / "models" / "anomaly_model.pkl"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Telemetry dataset not found at: {DATA_PATH}\n"
        "Run generate_telemetry.py first to create it."
    )

df = load_telemetry(str(DATA_PATH))
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

## 4. Dataset Overview

In [ ]:
print("=" * 55)
print("  TELEMETRY DATASET OVERVIEW")
print("=" * 55)
print(f"  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")
print()
print("  Column dtypes:")
for col, dtype in df.dtypes.items():
    print(f"    {col:<25} {str(dtype):<15}")
if "timestamp" in df.columns:
    print()
    print(f"  Time range:")
    print(f"    Start : {df['timestamp'].min()}")
    print(f"    End   : {df['timestamp'].max()}")
print("=" * 55)

In [ ]:
print("First 5 rows:")
df.head()

In [ ]:
print("Last 5 rows:")
df.tail()

## 5. Data Quality Analysis

Before training any model it is important to verify the dataset is complete and consistent. We check for:

- **Missing values** — nulls in any telemetry channel would silently corrupt the StandardScaler and IsolationForest.
- **Duplicate rows** — exact duplicates (same timestamp and all values) may indicate data ingestion errors.
- **Numeric validity** — all ML feature columns must be real-valued floats.

In [ ]:
# ── Missing values ───────────────────────────────────────────────────────────
missing = df.isnull().sum()
total_missing = int(missing.sum())

# ── Duplicates ────────────────────────────────────────────────────────────────
n_duplicates = int(df.duplicated().sum())

# ── Non-finite values in feature columns ────────────────────────────────────
non_finite = {}
for col in FEATURE_COLS:
    if col in df.columns:
        n_inf = int((~np.isfinite(df[col].astype(float))).sum())
        if n_inf > 0:
            non_finite[col] = n_inf

print("=" * 55)
print("  DATA QUALITY REPORT")
print("=" * 55)
print(f"  Total rows          : {len(df):,}")
print(f"  Missing values      : {total_missing} ",
      "✅" if total_missing == 0 else "⚠️  ACTION REQUIRED")
print(f"  Duplicate rows      : {n_duplicates} ",
      "✅" if n_duplicates == 0 else "⚠️  ACTION REQUIRED")
print(f"  Non-finite features : {len(non_finite)} ",
      "✅" if len(non_finite) == 0 else f"⚠️  {non_finite}")
print()

if total_missing > 0:
    print("  Missing values per column:")
    print(missing[missing > 0].to_string())
else:
    print("  All columns are complete — no missing values detected.")

print("=" * 55)

## 6. Descriptive Statistics

Summary statistics give us a baseline understanding of each telemetry channel's normal operating envelope. Key observations:

- **temperature** nominally ranges from −5 °C to 30 °C; values far outside this band signal thermal anomalies.
- **battery_voltage** is regulated near 28 V; significant deviation (under 27 V or over 29.5 V) indicates a battery fault.
- **power_consumption** varies with active subsystems (55–145 W nominal).
- **radiation_level** reflects the interplanetary background (0.10–1.30 mSv/h); elevated readings suggest solar particle events.
- **signal_strength** is negative dBm — values more negative than −82 dBm indicate signal fade.
- **fuel_level** depletes monotonically; sudden drops may indicate leaks or unplanned burns.
- **solar_output** depends on solar angle and eclipse; low values during expected illumination warrant investigation.

In [ ]:
CHANNEL_META = {
    "temperature":       ("Temperature",        "°C"),
    "battery_voltage":   ("Battery Voltage",     "V"),
    "power_consumption": ("Power Consumption",   "W"),
    "radiation_level":   ("Radiation Level",     "mSv/h"),
    "signal_strength":   ("Signal Strength",     "dBm"),
    "fuel_level":        ("Fuel Level",          "%"),
    "solar_output":      ("Solar Output",        "W"),
}

present_features = [c for c in FEATURE_COLS if c in df.columns]
stats = df[present_features].describe().T
stats.insert(0, "unit", [CHANNEL_META.get(c, (c, ""))[1] for c in stats.index])
stats.columns.name = None
pd.options.display.float_format = "{:.4f}".format
stats

## 7. Telemetry Time-Series Visualisation

Time-series plots reveal the temporal structure of each channel, including gradual trends (fuel depletion), periodic patterns (eclipse-driven solar drops), and the injected anomaly spikes that the IsolationForest must detect.

In [ ]:
COLORS = [
    "#3b82f6", "#22c55e", "#f59e0b",
    "#ef4444", "#a855f7", "#06b6d4", "#f97316",
]

time_col = "timestamp" if "timestamp" in df.columns else None
x_vals   = df[time_col] if time_col else df.index

fig, axes = plt.subplots(
    len(present_features), 1,
    figsize=(16, 3 * len(present_features)),
    sharex=True,
)
fig.suptitle(
    "AstraGuard AI — Simulated Spacecraft Telemetry (Full Mission Timeline)",
    fontsize=13, fontweight="bold", color="#e2e8f0", y=1.001,
)

for ax, col, color in zip(axes, present_features, COLORS):
    label, unit = CHANNEL_META.get(col, (col, ""))
    ax.plot(x_vals, df[col], color=color, linewidth=0.8, alpha=0.9)
    ax.set_ylabel(f"{label}\n({unit})", fontsize=8, color="#94a3b8")
    ax.grid(True, alpha=0.3)
    ax.tick_params(colors="#64748b", labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor("#1e3a5f")

if time_col:
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=7)

axes[-1].set_xlabel("Mission Time", fontsize=9, color="#94a3b8")
fig.tight_layout()
plt.show()

## 8. Feature Preparation

The ML model uses **all seven numeric telemetry channels** as input features. We reuse the `FEATURE_COLS` list defined in `src/anomaly_detector.py` to ensure the notebook is always consistent with the production code.

**Excluded columns:**
- `timestamp` — a datetime index, not a numeric sensor reading. Including it would cause the model to learn temporal position rather than telemetry patterns.
- `anomaly_score`, `anomaly_flag`, `label` — these are model *outputs*, not inputs. Including them would cause data leakage.

No manual feature engineering is applied. The IsolationForest operates directly on the scaled sensor values.

In [ ]:
print("Feature columns used by the AstraGuard AI pipeline:")
print()
for i, col in enumerate(FEATURE_COLS, 1):
    label, unit = CHANNEL_META.get(col, (col, ""))
    present = "✅" if col in df.columns else "❌ MISSING"
    print(f"  {i}. {col:<25} ({unit:<6})  {present}")

X = df[present_features].copy()
print()
print(f"Feature matrix shape: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Any NaN: {X.isnull().any().any()}")

## 9. Feature Scaling

The seven telemetry channels have very different magnitudes and units — temperature in °C spans roughly −25 to 85, while power consumption in Watts spans 30 to 270. Without scaling, IsolationForest's random partitioning would be dominated by the highest-magnitude channels, making the model insensitive to anomalies in low-magnitude channels.

**StandardScaler** transforms each feature to zero mean and unit variance:
$$z = \frac{x - \mu}{\sigma}$$

After scaling, all channels contribute equally to the anomaly score. The scaler is fitted on the full training dataset and applied inside the `sklearn.pipeline.Pipeline` so inference always uses the same transformation.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Demonstrate scaling on the raw feature matrix
demo_scaler = StandardScaler()
X_scaled    = demo_scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=present_features)

print("Scaled feature statistics (should be ≈ mean=0.0, std=1.0 per column):")
print()
summary = X_scaled_df.agg(["mean", "std"]).round(4)
print(summary.to_string())
print()
print("Note: In production, this scaler is embedded inside the sklearn Pipeline")
print("and is never fit separately — scaling happens automatically at predict time.")

## 10. IsolationForest Architecture

### How IsolationForest Works

IsolationForest is an **unsupervised anomaly detection** algorithm. It does not require labelled examples of anomalies — it learns the structure of normal data and identifies observations that deviate from it.

**Core concept — isolation by random partitioning:**
1. An ensemble of random binary trees is built by selecting a random feature and a random split value within its range.
2. Anomalous observations are isolated more quickly (fewer splits needed) because they are sparse and far from the bulk of the data.
3. The **anomaly score** (decision function) measures average path length across all trees.
   - More negative score → more anomalous.
   - Near zero / positive → normal behaviour.
4. The **anomaly flag** (`predict`) returns `-1` for anomalies and `+1` for normal observations.

### AstraGuard AI Pipeline Configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `n_estimators` | 200 | More trees → more stable, lower-variance scores |
| `contamination` | 0.05 | Matches the known ≈5% anomaly injection rate |
| `max_features` | 1.0 | All features used per tree |
| `random_state` | 42 | Reproducibility |
| Scaler | StandardScaler | Prevents high-magnitude channels from dominating |

In [ ]:
from anomaly_detector import build_pipeline, IF_PARAMS

print("IsolationForest hyperparameters (from src/anomaly_detector.py):")
print()
for k, v in IF_PARAMS.items():
    print(f"  {k:<20} = {v}")

print()
demo_pipeline = build_pipeline()
print("Pipeline structure:")
print(demo_pipeline)

## 11. Load the Production Model

The production model was trained by `train_anomaly_model.py` and serialised to `models/anomaly_model.pkl` using `joblib`. We load it here to avoid unnecessary retraining and to ensure the notebook uses exactly the same model as the Streamlit dashboard.

If the model file is unavailable, we fall back to training a fresh pipeline on the loaded data.

In [ ]:
if MODEL_PATH.exists():
    pipeline = load_model(str(MODEL_PATH))
    print(f"✅ Production model loaded from: {MODEL_PATH}")
    print()
    iforest = pipeline.named_steps["iforest"]
    scaler  = pipeline.named_steps["scaler"]
    print(f"  Pipeline steps    : {list(pipeline.named_steps.keys())}")
    print(f"  Scaler means      : {scaler.mean_.round(4)}")
    print(f"  IsolationForest   : n_estimators={iforest.n_estimators}, "
          f"contamination={iforest.contamination}")
    print(f"  Trees in forest   : {len(iforest.estimators_)}")
    print(f"  Feature count     : {iforest.n_features_in_}")
else:
    print(f"⚠️  Model not found at {MODEL_PATH} — training fresh pipeline.")
    from anomaly_detector import train
    pipeline, _ = train(df)
    print("   Fresh pipeline trained on loaded telemetry data.")
    print("   (Run train_anomaly_model.py to save the model for future use.)")

## 12. Generate Anomaly Predictions

We apply the loaded pipeline to the full telemetry dataset using `score_dataframe()` from `src/anomaly_detector.py`. This appends three new columns:

| Column | Type | Description |
|--------|------|-------------|
| `anomaly_score` | float | IsolationForest decision score (more negative = more anomalous) |
| `anomaly_flag` | int | `−1` = anomaly, `+1` = normal |
| `label` | str | Human-readable `"Anomaly"` / `"Normal"` |

In [ ]:
scored_df = score_dataframe(pipeline, df)

summary = detection_summary(scored_df)

print("=" * 50)
print("  ANOMALY DETECTION SUMMARY")
print("=" * 50)
print(f"  Total observations  : {summary['total_records']:,}")
print(f"  Normal              : {summary['normal_count']:,}")
print(f"  Anomalous           : {summary['anomaly_count']:,}")
print(f"  Anomaly rate        : {summary['anomaly_rate_pct']:.2f}%")
print(f"  Min anomaly score   : {summary['min_score']:.6f}")
print(f"  Max anomaly score   : {summary['max_score']:.6f}")
print("=" * 50)
print()
print("Sample output (5 anomaly rows):")
cols_show = ["timestamp", "anomaly_score", "anomaly_flag", "label"] + present_features[:3]
scored_df[scored_df["label"] == "Anomaly"].head(5)[cols_show]

## 13. Anomaly Analysis

We examine the statistical profile of anomalous versus normal observations to understand what the IsolationForest learned to flag.

In [ ]:
normal_df = scored_df[scored_df["label"] == "Normal"]
anomaly_df = scored_df[scored_df["label"] == "Anomaly"]

print("Mean values — Normal vs Anomalous observations:")
print()
comparison = pd.DataFrame({
    "Normal (mean)": normal_df[present_features].mean(),
    "Anomaly (mean)": anomaly_df[present_features].mean(),
    "Unit": [CHANNEL_META.get(c, (c, ""))[1] for c in present_features],
})
comparison["Δ (Anomaly − Normal)"] = (
    comparison["Anomaly (mean)"] - comparison["Normal (mean)"]
).round(4)
comparison = comparison.round(4)
print(comparison.to_string())

## 14. Visualise Anomalies on Telemetry Channels

For each telemetry channel we overlay normal readings (blue) and anomalous readings (red) on the mission timeline. This reveals which channels are most visually distinguishable between normal and anomalous states.

In [ ]:
time_col = "timestamp" if "timestamp" in scored_df.columns else None

n_cols   = 2
n_rows   = (len(present_features) + 1) // n_cols
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(16, 4 * n_rows),
)
axes = axes.flatten()

norm_mask = scored_df["label"] == "Normal"
anom_mask = scored_df["label"] == "Anomaly"

for i, col in enumerate(present_features):
    ax = axes[i]
    label, unit = CHANNEL_META.get(col, (col, ""))
    x_n = scored_df.loc[norm_mask, time_col] if time_col else scored_df.index[norm_mask]
    x_a = scored_df.loc[anom_mask, time_col] if time_col else scored_df.index[anom_mask]

    ax.scatter(x_n, scored_df.loc[norm_mask, col],
               s=3, alpha=0.4, color="#3b82f6", label="Normal", zorder=2)
    ax.scatter(x_a, scored_df.loc[anom_mask, col],
               s=20, alpha=0.85, color="#ef4444", label="Anomaly",
               marker="x", linewidths=1.2, zorder=3)

    ax.set_title(f"{label}", fontsize=9, color="#e2e8f0", pad=6)
    ax.set_ylabel(f"{unit}", fontsize=8)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=7, framealpha=0.3, loc="upper right",
              labelcolor="#e2e8f0", facecolor="#111827")
    for spine in ax.spines.values():
        spine.set_edgecolor("#1e3a5f")
    if time_col:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, fontsize=6)

# Hide any unused axes
for j in range(len(present_features), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    "AstraGuard AI — Normal vs Anomalous Readings per Telemetry Channel",
    fontsize=12, fontweight="bold", color="#e2e8f0", y=1.01,
)
fig.tight_layout()
plt.show()

## 15. Anomaly Score Distribution

The IsolationForest decision score provides a continuous measure of anomalousness. More negative values indicate observations that were isolated more quickly — consistent with being statistically unusual. The threshold between `−1` (anomaly) and `+1` (normal) flags is automatically determined by the `contamination` parameter during training.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: histogram of scores by label ───────────────────────────────────────
bins = 60
ax1.hist(
    scored_df.loc[norm_mask, "anomaly_score"],
    bins=bins, alpha=0.75, color="#3b82f6", label="Normal", density=True,
)
ax1.hist(
    scored_df.loc[anom_mask, "anomaly_score"],
    bins=bins, alpha=0.85, color="#ef4444", label="Anomaly", density=True,
)
ax1.axvline(0, color="#f59e0b", linewidth=1.2, linestyle="--", label="Score = 0")
ax1.set_xlabel("IsolationForest Decision Score", fontsize=9)
ax1.set_ylabel("Density", fontsize=9)
ax1.set_title("Anomaly Score Distribution", fontsize=10, color="#e2e8f0")
ax1.legend(fontsize=8, framealpha=0.3, facecolor="#111827", labelcolor="#e2e8f0")
ax1.grid(True, alpha=0.25)

# ── Right: score over time ────────────────────────────────────────────────────
x_time = scored_df[time_col] if time_col else scored_df.index
ax2.scatter(
    x_time[norm_mask], scored_df.loc[norm_mask, "anomaly_score"],
    s=3, alpha=0.4, color="#3b82f6", label="Normal",
)
ax2.scatter(
    x_time[anom_mask], scored_df.loc[anom_mask, "anomaly_score"],
    s=18, alpha=0.85, color="#ef4444", label="Anomaly",
    marker="x", linewidths=1.2, zorder=3,
)
ax2.axhline(0, color="#f59e0b", linewidth=1.0, linestyle="--", alpha=0.7)
ax2.set_xlabel("Mission Time", fontsize=9)
ax2.set_ylabel("Decision Score", fontsize=9)
ax2.set_title("Anomaly Score Over Mission Timeline", fontsize=10, color="#e2e8f0")
ax2.legend(fontsize=8, framealpha=0.3, facecolor="#111827", labelcolor="#e2e8f0")
ax2.grid(True, alpha=0.25)
if time_col:
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, fontsize=7)

for ax in (ax1, ax2):
    for spine in ax.spines.values():
        spine.set_edgecolor("#1e3a5f")

fig.suptitle(
    "IsolationForest Decision Score Analysis",
    fontsize=12, fontweight="bold", color="#e2e8f0",
)
fig.tight_layout()
plt.show()

print()
score_summary = scored_df.groupby("label")["anomaly_score"].agg(["mean","median","min","max"]).round(6)
print("Score statistics by class:")
print(score_summary.to_string())

## 16. Top Anomalous Observations

The records with the most negative decision scores represent the most statistically unusual telemetry readings detected by the model. These would typically be the first records an operator reviews.

In [ ]:
top_n = 10
top_anomalies = (
    scored_df[scored_df["label"] == "Anomaly"]
    .sort_values("anomaly_score", ascending=True)
    .head(top_n)
)

display_cols = (["timestamp"] if time_col else []) + ["anomaly_score"] + present_features
print(f"Top {top_n} most anomalous telemetry readings (lowest decision score first):")
print()
pd.options.display.float_format = "{:.4f}".format
top_anomalies[display_cols].reset_index(drop=True)

## 17. Feature Correlation

Examining correlations between telemetry channels helps us understand whether some anomalies are driven by co-varying signals. High correlations between certain channels may indicate shared physical dependencies (e.g., a drop in `solar_output` often accompanies reduced `battery_voltage` if the spacecraft enters eclipse).

In [ ]:
import matplotlib.colors as mcolors

corr = df[present_features].corr()

fig, ax = plt.subplots(figsize=(9, 7))

cmap = matplotlib.colormaps.get_cmap("RdBu").resampled(256)
im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect="auto")

channel_labels = [CHANNEL_META.get(c, (c, ""))[0] for c in present_features]
ax.set_xticks(range(len(present_features)))
ax.set_yticks(range(len(present_features)))
ax.set_xticklabels(channel_labels, rotation=40, ha="right", fontsize=8)
ax.set_yticklabels(channel_labels, fontsize=8)

for i in range(len(present_features)):
    for j in range(len(present_features)):
        val = corr.values[i, j]
        txt_color = "#e2e8f0" if abs(val) > 0.4 else "#1f2328"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=7, color=txt_color)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(colors="#94a3b8", labelsize=7)
ax.set_title(
    "Telemetry Channel Correlation Matrix",
    fontsize=11, color="#e2e8f0", pad=12,
)
for spine in ax.spines.values():
    spine.set_edgecolor("#1e3a5f")

fig.tight_layout()
plt.show()

## 18. Mission Risk Engine Integration

The anomaly detector output feeds directly into the **Mission Risk Engine** (`src/risk_engine.py`), which translates raw ML flags into an explainable **Mission Risk Index (MRI)** on a 0–100 scale.

### Risk Engine Architecture

For each telemetry row, the risk engine:
1. **Threshold Exceedance** — compares each channel against a pre-defined safe operating envelope and produces a 0–100 channel score.
2. **Anomaly Boost** — if IsolationForest also flagged the row, all non-zero channel scores are multiplied by `ANOMALY_BOOST = 1.35`, rewarding agreement between the rule-based and ML signals.
3. **Weighted Sum** — channel scores are combined using operational criticality weights (battery: 0.25, temperature: 0.20, power: 0.18, radiation: 0.15, signal: 0.12, solar: 0.10) and scaled to 0–100.

| MRI Range | Risk Level | Meaning |
|-----------|------------|---------|
| 0–30 | LOW | Nominal operations |
| 31–60 | MEDIUM | Operator attention warranted |
| 61–80 | HIGH | Immediate investigation required |
| 81–100 | CRITICAL | Mission-threatening condition |

In [ ]:
from risk_engine import (
    assess_dataframe,
    assess_row,
    batch_summary,
    ENVELOPES,
    RISK_CHANNELS,
)

risk_df = assess_dataframe(scored_df)

risk_summary = batch_summary(risk_df)
level_counts = risk_df["risk_level"].value_counts()

print("=" * 55)
print("  MISSION RISK INDEX SUMMARY")
print("=" * 55)
print(f"  Current MRI (latest reading) : {risk_summary['current_risk_score']}/100  "
      f"({risk_summary['current_risk_level']})")
print(f"  Peak MRI                     : {risk_summary['max_risk_score']}/100")
print(f"  Mean MRI                     : {risk_summary['mean_risk_score']}/100")
print()
print("  Risk level distribution:")
for level in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
    count = int(level_counts.get(level, 0))
    pct   = 100 * count / len(risk_df)
    bar   = "█" * int(pct / 2)
    print(f"    {level:<10} {count:>5}  ({pct:5.1f}%)  {bar}")
print("=" * 55)

In [ ]:
LEVEL_COLORS = {
    "LOW": "#22c55e",
    "MEDIUM": "#f59e0b",
    "HIGH": "#ef4444",
    "CRITICAL": "#7c3aed",
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: MRI over time ───────────────────────────────────────────────────────
x_plot = risk_df[time_col] if time_col else risk_df.index
colors_per_point = risk_df["risk_level"].map(LEVEL_COLORS).fillna("#64748b")
ax1.scatter(x_plot, risk_df["risk_score"],
            c=colors_per_point, s=5, alpha=0.7, zorder=2)
for threshold, color, label in [(81, "#7c3aed", "CRITICAL"),
                                  (61, "#ef4444", "HIGH"),
                                  (31, "#f59e0b", "MEDIUM")]:
    ax1.axhline(threshold, color=color, linewidth=0.8, linestyle="--", alpha=0.6)
ax1.set_xlabel("Mission Time", fontsize=9)
ax1.set_ylabel("Mission Risk Index", fontsize=9)
ax1.set_title("Mission Risk Index Over Time", fontsize=10, color="#e2e8f0")
ax1.set_ylim(-2, 105)
ax1.grid(True, alpha=0.2)
if time_col:
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, fontsize=7)

# ── Right: risk level distribution bar ───────────────────────────────────────
levels_ordered = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
counts = [int(level_counts.get(lvl, 0)) for lvl in levels_ordered]
bar_colors = [LEVEL_COLORS[lvl] for lvl in levels_ordered]
bars = ax2.bar(levels_ordered, counts, color=bar_colors, edgecolor="#0a0e1a", linewidth=0.8)
for bar, count in zip(bars, counts):
    if count > 0:
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
                 str(count), ha="center", va="bottom", fontsize=8, color="#94a3b8")
ax2.set_xlabel("Risk Level", fontsize=9)
ax2.set_ylabel("Number of Readings", fontsize=9)
ax2.set_title("Risk Level Distribution", fontsize=10, color="#e2e8f0")
ax2.grid(True, alpha=0.2, axis="y")

for ax in (ax1, ax2):
    for spine in ax.spines.values():
        spine.set_edgecolor("#1e3a5f")

fig.suptitle(
    "Mission Risk Index Analysis — AstraGuard AI",
    fontsize=12, fontweight="bold", color="#e2e8f0",
)
fig.tight_layout()
plt.show()

## 19. AI Mission Explanation

The final layer of AstraGuard AI generates a **natural-language explanation** for each detected anomaly using `src/ai_explanation.py`. This bridges the gap between raw ML output and operator-readable mission analysis.

We demonstrate the template-based (offline) backend on the most critical detected anomaly.

In [ ]:
from ai_explanation import ExplanationInput, generate_explanation

# Find the highest-risk anomalous reading
high_risk = (
    risk_df[risk_df["label"] == "Anomaly"]
    .sort_values("risk_score", ascending=False)
)

if len(high_risk) == 0:
    print("No anomalies found in this dataset.")
else:
    worst_row  = high_risk.iloc[0]
    assessment = assess_row(
        worst_row,
        anomaly_flag=int(worst_row["anomaly_flag"]),
        anomaly_score=float(worst_row["anomaly_score"]),
    )
    inp  = ExplanationInput.from_assessment(worst_row, assessment)
    text = generate_explanation(inp, backend="template")

    print("=" * 65)
    print("  AI MISSION ANALYSIS — Highest Risk Anomaly")
    print("=" * 65)
    print(f"  Timestamp   : {worst_row.get('timestamp', 'N/A')}")
    print(f"  Risk Score  : {assessment.risk_score}/100  ({assessment.risk_level})")
    print(f"  IF Score    : {assessment.anomaly_score:.6f}")
    print()
    print("  Generated Explanation:")
    print()
    # Wrap at ~70 chars for readable notebook output
    import textwrap
    for line in textwrap.wrap(text, width=70):
        print("  " + line)
    print("=" * 65)

## 20. Results Interpretation

### What the Anomaly Detector Found

The IsolationForest model, trained on 1,000 five-minute simulated telemetry readings, identified a subset of observations (approximately 5%, consistent with the configured `contamination` parameter) as statistically unusual compared to the learned normal distribution of spacecraft operations.

### What Anomalous Telemetry Means in This Prototype

In the context of this simulated dataset, anomalous readings correspond to injected fault scenarios:

- **Battery voltage drop** (below ~21 V) — simulates cell degradation or deep discharge.
- **Temperature spike** (above ~80 °C) — simulates thermal runaway or heater failure.
- **Power surge** (above ~250 W) — simulates a short-circuit or unplanned subsystem activation.
- **Radiation burst** (above ~5 mSv/h) — simulates a solar particle event.
- **Signal fade** (below ~−110 dBm) — simulates antenna misalignment or occultation.
- **Solar drop** (below ~20 W during illumination) — simulates panel obstruction.
- **Multivariate deviations** — combinations of subtle out-of-range readings across channels.

### Why Anomalies Should Be Investigated

In a real system, each anomalous reading would trigger an operator review because:
1. Early detection of hardware degradation reduces mission risk.
2. Single-channel anomalies may precede cascade failures.
3. Multivariate anomalies (no single channel out of range, but the *combination* is unusual) are invisible to threshold monitoring.

### Prototype Limitations

> ⚠️ This prototype operates on **entirely simulated telemetry** and is **not validated** against real mission data.

- **No ground truth labels** are used — this is an unsupervised system. Precision and recall cannot be honestly computed.
- **Fixed contamination** (0.05) assumes 5% of readings are anomalous. Real spacecraft operations have very different anomaly rates.
- **Threshold envelopes** in the risk engine are manually tuned to the simulated data, not derived from real operating specifications.
- **IsolationForest** does not distinguish between fault types — all anomalies are treated equally by the model (though the risk engine provides per-channel attribution).


## 21. Full Pipeline Architecture

The complete AstraGuard AI pipeline from raw telemetry to dashboard display:

```
┌─────────────────────────────────────────────────────────┐
│              ASTRAGUARD AI PIPELINE                     │
├─────────────────────────────────────────────────────────┤
│  📡  Simulated Telemetry CSV (data/telemetry.csv)       │
│      7 channels × 1,000 records @ 5-minute intervals   │
│                          │                             │
│  🔍  Data Quality & Exploration                         │
│      Missing value checks, descriptive statistics      │
│                          │                             │
│  ⚖️   Feature Preparation                               │
│      Select 7 numeric channels (exclude timestamp)     │
│                          │                             │
│  📐  StandardScaler                                     │
│      Zero-mean, unit-variance normalisation            │
│                          │                             │
│  🌲  IsolationForest (n=200, contamination=0.05)        │
│      Unsupervised random-partition anomaly detection   │
│                          │                             │
│  🚦  Anomaly Score  +  Anomaly Flag (-1/+1)             │
│                          │                             │
│  📊  Mission Risk Engine (src/risk_engine.py)           │
│      Threshold exceedance + anomaly boost              │
│      → Mission Risk Index 0–100  (LOW/MEDIUM/HIGH/CRIT)│
│                          │                             │
│  🤖  AI Explanation Engine (src/ai_explanation.py)      │
│      Natural-language mission analysis (template/LLM)  │
│                          │                             │
│  🛰️   AstraGuard AI Dashboard (app.py, Streamlit)       │
│      Live telemetry charts, risk gauge, Q&A interface  │
└─────────────────────────────────────────────────────────┘
```


## 22. Pipeline Summary Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: Anomaly rate ─────────────────────────────────────────────────────
ax = axes[0]
n_norm = summary["normal_count"]
n_anom = summary["anomaly_count"]
wedge_colors = ["#3b82f6", "#ef4444"]
wedges, texts, autotexts = ax.pie(
    [n_norm, n_anom],
    labels=["Normal", "Anomaly"],
    colors=wedge_colors,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.75,
    textprops={"color": "#e2e8f0", "fontsize": 9},
)
for at in autotexts:
    at.set_color("#0a0e1a")
    at.set_fontsize(9)
ax.set_title(
    f"Detection Split\n({summary['total_records']:,} total readings)",
    color="#e2e8f0", fontsize=9,
)

# ── Panel 2: Risk level bar ───────────────────────────────────────────────────
ax = axes[1]
lvl_order = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
lvl_counts = [int(level_counts.get(l, 0)) for l in lvl_order]
lvl_colors = [LEVEL_COLORS[l] for l in lvl_order]
brs = ax.barh(lvl_order, lvl_counts, color=lvl_colors, edgecolor="#0a0e1a")
for br, cnt in zip(brs, lvl_counts):
    if cnt > 0:
        ax.text(cnt + 3, br.get_y() + br.get_height() / 2,
                str(cnt), va="center", fontsize=8, color="#94a3b8")
ax.set_xlabel("Number of Readings", fontsize=8)
ax.set_title("Risk Level Distribution\n(all 1,000 readings)", color="#e2e8f0", fontsize=9)
ax.grid(True, alpha=0.2, axis="x")
for spine in ax.spines.values():
    spine.set_edgecolor("#1e3a5f")

# ── Panel 3: Channel weight contribution ─────────────────────────────────────
ax = axes[2]
weights = {ch: ENVELOPES[ch].weight for ch in RISK_CHANNELS}
labels_w = [CHANNEL_META.get(ch, (ch, ""))[0] for ch in weights]
vals_w   = list(weights.values())
colors_w = ["#3b82f6", "#22c55e", "#f59e0b", "#ef4444", "#a855f7", "#06b6d4"]
bars_w = ax.bar(labels_w, vals_w, color=colors_w[:len(vals_w)], edgecolor="#0a0e1a")
for bar, val in zip(bars_w, vals_w):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.004,
            f"{val:.2f}", ha="center", fontsize=7, color="#94a3b8")
ax.set_ylabel("MRI Weight", fontsize=8)
ax.set_title("Channel Criticality Weights\n(Mission Risk Engine)", color="#e2e8f0", fontsize=9)
ax.set_ylim(0, 0.32)
ax.grid(True, alpha=0.2, axis="y")
plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha="right", fontsize=7)
for spine in ax.spines.values():
    spine.set_edgecolor("#1e3a5f")

fig.suptitle(
    "AstraGuard AI — Pipeline Results Summary",
    fontsize=12, fontweight="bold", color="#e2e8f0",
)
fig.tight_layout()
plt.show()

## 23. Conclusion

---

### AstraGuard AI — IBM AI Builders Challenge 2026

This notebook has demonstrated the complete machine-learning pipeline at the core of AstraGuard AI, from raw simulated telemetry ingestion through to natural-language mission analysis:

**What was demonstrated:**

1. **Exploratory Data Analysis** — 1,000 five-minute simulated telemetry readings across seven spacecraft channels were inspected, verified for quality, and visualised on the mission timeline.

2. **Unsupervised Anomaly Detection** — An `IsolationForest` model (200 trees, 5% contamination) trained within a `StandardScaler → IsolationForest` sklearn Pipeline identified statistically unusual readings without requiring any labelled fault examples.

3. **Explainable Risk Scoring** — The Mission Risk Engine translated ML anomaly flags into a transparent 0–100 Mission Risk Index by combining per-channel threshold exceedance with operational criticality weights, producing audit-able, traceable risk scores.

4. **Natural-Language Explanation** — The AI Explanation Engine generated professional operator-readable mission analyses for each detected anomaly, clearly attributing risk to specific telemetry channels and recommending targeted investigation steps.

**AI Approach Highlights:**

- Unsupervised learning enables detection of novel fault patterns with no labelled training data.
- The pipeline architecture separates concerns cleanly: detection (ML) → scoring (rules + ML) → explanation (template/LLM).
- All components are designed to be extended: the template backend can be upgraded to OpenAI or IBM Granite at any time by changing one parameter.

**Prototype Scope:**

> This system is a prototype built on simulated telemetry for the IBM AI Builders Challenge.  
> It is **not** a certified aerospace system and must **not** be used for real mission operations.

---

*AstraGuard AI — Advance Space Exploration with Artificial Intelligence*  
*IBM AI Builders Challenge, August 2026*
